# Try ReFactX

## Setup
Create a `.env` file in the notebooks folder adding the following variable:
```
HTTP_BASE_URL="http://{user}:{password}@{host}:{port}/{dbname}"
```
Then append `/tablename` for using a specific db table.

(you can also set the HTTP_BASE_URL in the environment)

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
from dotenv import load_dotenv

import torch
import time
from transformers.generation.logits_process import LogitsProcessorList
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer
from transformers import AutoProcessor, AutoModelForImageTextToText

import refactx

/opt/conda/envs/trl/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/conda/envs/trl/lib/python3.14/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [3]:
load_dotenv()
HTTP_BASE_URL = os.environ.get('HTTP_BASE_URL')

In [4]:
tablename = 'qwen36'
HTTP_URL = f'{HTTP_BASE_URL}/{tablename}'

In [5]:
#MODEL = 'Qwen/Qwen3.5-2B'
MODEL = 'Qwen/Qwen3.5-4B'
IS_VLM = True

In [6]:
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
DEVICE

'cuda'

In [7]:
if IS_VLM:
    processor = AutoProcessor.from_pretrained(MODEL)
    model = AutoModelForImageTextToText.from_pretrained(MODEL, device_map='auto')
    tokenizer = processor
else:
    tokenizer = AutoTokenizer.from_pretrained(MODEL)
    model = AutoModelForCausalLM.from_pretrained(MODEL, device_map='auto')

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 723/723 [00:02<00:00, 302.49it/s]


In [8]:
index = refactx.load_index(
    HTTP_URL, 
    #tokenizer,
    #configkey=-200,
    #cache='simple'
)

Applying index config...


In [9]:
index.get_config()

Applying index config...


{'switch_parameter': 7, 'rootkey': -100, 'tokenizer_name': 'Qwen/Qwen3.6-27B'}

In [10]:
streamer = TextStreamer(tokenizer)

            "Rules:\n"
            "- First you identify the entities related to the claim.'\n"
            "- Then you start generating PRO and CONTRA facts.'\n"
            "- Every generated statement must start with 'Fact:'\n"
            "- The 'Fact:' command enables constrained generation from Wikidata.'\n"
            "- Facts must be meaningfully connected to the claim.\n"
            "- You generate facts related to the entities in the claim.\n"

In [11]:
PROMPT_TEMPLATE_1 = [
    {
        "role": "system",
        "content": (
            "You are a fact checking system over facts from Wikidata.\n\n"
            "Task:\n"
            "Given a claim you gather related facts.\n"
            "Try to stop as soon as you have enough information for fact checking.\n"
            'Each fact MUST start with "Fact:".\n'
            "Example:\n\n"
            "Given claim: Barack Obama was president of the USA."
            "Facts:\n"
            "Fact: <Barack Obama> <position held> <President of the United States> .\n"
            "Fact: <Barack Obama> <country of citizenship> <United States> .\n"
            "Fact: <Barack Obama> <child> <Malia Obama (daughter of former US President Barack Obama and Michelle Obama)> .\n"
            "Fact: <Barack Obama> <candidacy in election> <2012 United States presidential election in Missouri (election in Missouri)> .\n"
            "Fact: <Barack Obama> <candidacy in election> <2008 United States presidential election (56th quadrennial U.S. presidential election)> .\n"
        ),
    }
]

In [12]:
THINKING = False

claim = 'Claim: Barack Obama is the president of USA.'
claim = 'Claim: Elon Musk founded Tesla.'
#claim = 'Who is older? Brad Pitt or Johnny Depp?'

prompted_texts = [refactx.apply_prompt_template(tokenizer,
                                                question=claim,
                                                prompt_template=PROMPT_TEMPLATE_1,
                                                enable_thinking=THINKING)]

In [13]:
inputs = tokenizer.tokenizer(prompted_texts, return_tensors='pt', padding=True, padding_side='right')
inputs = inputs.to(model.device)
print(inputs['input_ids'].shape)

torch.Size([1, 226])


In [14]:
model.device

device(type='cuda', index=0)

In [15]:
# no need for num_beams=1
#refactx.patch_model(model)

In [16]:
num_beams = 1
num_batches = 1

auto_streamer = streamer if num_beams == 1 else None

In [17]:
constrained_processor = refactx.get_constrained_logits_processor(tokenizer, index, num_beams, num_batches)

In [18]:
logits_processor_list = constrained_processor

model.eval()
start = time.time()

with torch.no_grad():
    out = model.generate(
        **inputs,
        logits_processor=logits_processor_list,
        max_new_tokens=200,
        streamer = auto_streamer,
        do_sample = False,
        temperature = None,
        top_k=None,
        num_beams=num_beams,
        num_return_sequences=num_beams,
        use_cache=True,
        top_p=None,
        min_p=None,
    )

print('Elapsed', time.time() - start)

Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


<|im_start|>system
You are a fact checking system over facts from Wikidata.

Task:
Given a claim you gather related facts.
Try to stop as soon as you have enough information for fact checking.
Each fact MUST start with "Fact:".
Example:

Given claim: Barack Obama was president of the USA.Facts:
Fact: <Barack Obama> <position held> <President of the United States> .
Fact: <Barack Obama> <country of citizenship> <United States> .
Fact: <Barack Obama> <child> <Malia Obama (daughter of former US President Barack Obama and Michelle Obama)> .
Fact: <Barack Obama> <candidacy in election> <2012 United States presidential election in Missouri (election in Missouri)> .
Fact: <Barack Obama> <candidacy in election> <2008 United States presidential election (56th quadrennial U.S. presidential election)> .<|im_end|>
<|im_start|>user
Claim: Elon Musk founded Tesla.<|im_end|>
<|im_start|>assistant
<think>

</think>

Fact: <Elon Musk> <employer> <Tesla, Inc.> .
Fact: <Elon Musk> <date of birth> <1971-0

In [19]:
_from = len(inputs.input_ids[0]) # 0
for i in range(out.shape[0]):
    print('-'*30, sum(out[i][_from:]), len(out[i][_from:]))
    prompt_1_response = tokenizer.decode(out[i][_from:])
    #print(prompt_1_response)
    break

------------------------------ tensor(1724334, device='cuda:0') 200


In [20]:
prompt_1_response = prompt_1_response[len('Fact:\n')+1:]
prompt_1_response = f"Given claim: {claim}\nGiven facts:\n{prompt_1_response}"
print(prompt_1_response)

Given claim: Claim: Elon Musk founded Tesla.
Given facts:
Elon Musk> <employer> <Tesla, Inc.> .
Fact: <Elon Musk> <date of birth> <1971-06-28T00:00:00Z> .
Fact: <Tesla, Inc.> <founded by> <Martin Eberhard> .
Fact: <Tesla, Inc.> <founded by> <Marc Tarpenning> .
Fact: <Tesla, Inc.> <country of origin> <United States> .
Fact: <Tesla, Inc.> <country> <United States> .
Fact: <Tesla, Inc.> <headquarters location> <Austin, Texas> .
Fact: <Tesla, Inc.> <stock exchange> <Nasdaq> .
Fact: <Tesla, Inc.> <stock exchange> <London Stock Exchange> .
Fact: <Tesla, Inc.> <stock


## Second Pass

In [21]:
triple_lst = refactx.get_constrained_states()[0][0].generated_triples
triple_lst = list(map(tokenizer.decode, triple_lst))
triple_lst

[' <Elon Musk> <employer> <Tesla, Inc.> .',
 ' <Elon Musk> <date of birth> <1971-06-28T00:00:00Z> .',
 ' <Tesla, Inc.> <founded by> <Martin Eberhard> .',
 ' <Tesla, Inc.> <founded by> <Marc Tarpenning> .',
 ' <Tesla, Inc.> <country of origin> <United States> .',
 ' <Tesla, Inc.> <country> <United States> .',
 ' <Tesla, Inc.> <headquarters location> <Austin, Texas> .',
 ' <Tesla, Inc.> <stock exchange> <Nasdaq> .',
 ' <Tesla, Inc.> <stock exchange> <London Stock Exchange> .']

In [22]:
index2 = refactx.load_index(triple_lst, tokenizer=tokenizer)

100%|██████████| 1/1 [00:00<00:00, 1219.98it/s]


In [24]:
PROMPT_TEMPLATE_2 = [
    {
        "role": "system",
        "content": (
            "You are a fact checking system over fact from Wikidata.\n\n"
            "Task:\n"
            "Given a claim and related facts, you organize the facts between supporting (PRO) and contradiction (CONTRA).\n"
            'Each fact MUST start with "Fact:".\n'
            "After that you conclude if the claim is correct or not.\n"
            "Example:\n\n"
            "Given claim: Barack Obama was president of the USA."
            "Given facts:\n"
            "Fact: <Barack Obama> <position held> <President of the United States> .\n"
            "Fact: <Barack Obama> <country of citizenship> <United States> .\n"
            "Fact: <Barack Obama> <employer> <New York Public Interest Research Group> .\n"
            "Fact: <Barack Obama> <child> <Malia Obama (daughter of former US President Barack Obama and Michelle Obama)> .\n"
            "Fact: <Barack Obama> <candidacy in election> <2012 United States presidential election in Missouri (election in Missouri)> .\n"
            "Fact: <Barack Obama> <candidacy in election> <2008 United States presidential election (56th quadrennial U.S. presidential election)> .\n"
            "Fact: <Barack Obama> <occupation> <political writer (profession)> ."
            "\n"
            "PRO:\n"
            "Fact: <Barack Obama> <position held> <President of the United States> .\n"
            "Fact: <Barack Obama> <country of citizenship> <United States> .\n\n"
            "Fact: <Barack Obama> <child> <Malia Obama (daughter of former US President Barack Obama and Michelle Obama)> .\n"
            "Fact: <Barack Obama> <candidacy in election> <2008 United States presidential election (56th quadrennial U.S. presidential election)> .\n"
            "Fact: <Barack Obama> <candidacy in election> <2012 United States presidential election in Missouri (election in Missouri)> .\n\n"
            "CONTRA:\n"
            "Fact: <Barack Obama> <employer> <New York Public Interest Research Group> .\n"
            "Fact: <Barack Obama> <occupation> <political writer (profession)> .\n"
            "REASONING:\n"
            "The claim states that Barack Obama held the office of President of the United States. "
            "Several provided facts directly support this, including his position held as President of the United States and his U.S. citizenship, "
            "which is consistent with eligibility and context for the claim. "
            "Other facts, such as employment history or occupation as a political writer, are not directly relevant to presidency and therefore "
            "do not contradict the claim. No provided fact explicitly states that he did not hold the presidency. "
            "Thus, the evidence overall supports the claim.\n\n"
            "CONCLUSION:\n"
            "The claim is SUPPORTED.\n"
        ),
    }
]

In [25]:
THINKING = False
prompted_texts_2 = [refactx.apply_prompt_template(tokenizer,
                                                question=prompt_1_response,
                                                prompt_template=PROMPT_TEMPLATE_2,
                                                enable_thinking=THINKING)]

In [26]:
inputs = tokenizer.tokenizer(prompted_texts_2, return_tensors='pt', padding=True, padding_side='right')
inputs = inputs.to(model.device)
print(inputs['input_ids'].shape)

torch.Size([1, 782])


In [27]:
model.device

device(type='cuda', index=0)

In [28]:
# no need for num_beams=1
#refactx.patch_model(model)

In [29]:
num_beams = 1
num_batches = 1

auto_streamer = streamer if num_beams == 1 else None

In [30]:
constrained_processor = refactx.get_constrained_logits_processor(tokenizer, index2, num_beams, num_batches)

In [31]:
logits_processor_list = constrained_processor

model.eval()
start = time.time()

with torch.no_grad():
    out = model.generate(
        **inputs,
        logits_processor=logits_processor_list,
        max_new_tokens=2000,
        streamer = auto_streamer,
        do_sample = False,
        temperature = None,
        top_k=None,
        num_beams=num_beams,
        num_return_sequences=num_beams,
        use_cache=True,
        top_p=None,
        min_p=None,
    )

print('Elapsed', time.time() - start)

Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


<|im_start|>system
You are a fact checking system over fact from Wikidata.

Task:
Given a claim and related facts, you organize the facts between supporting (PRO) and contradiction (CONTRA).
Each fact MUST start with "Fact:".
After that you conclude if the claim is correct or not.
Example:

Given claim: Barack Obama was president of the USA.Given facts:
Fact: <Barack Obama> <position held> <President of the United States> .
Fact: <Barack Obama> <country of citizenship> <United States> .
Fact: <Barack Obama> <employer> <New York Public Interest Research Group> .
Fact: <Barack Obama> <child> <Malia Obama (daughter of former US President Barack Obama and Michelle Obama)> .
Fact: <Barack Obama> <candidacy in election> <2012 United States presidential election in Missouri (election in Missouri)> .
Fact: <Barack Obama> <candidacy in election> <2008 United States presidential election (56th quadrennial U.S. presidential election)> .
Fact: <Barack Obama> <occupation> <political writer (profess

### Visualize ReFactX output

In [32]:
_from = len(inputs.input_ids[0]) # 0
for i in range(out.shape[0]):
    print('-'*30, sum(out[i][_from:]), len(out[i][_from:]))
    print(tokenizer.decode(out[i][_from:]))

------------------------------ tensor(3136799, device='cuda:0') 312
PRO:
Fact: <Elon Musk> <employer> <Tesla, Inc.> .
Fact: <Elon Musk> <date of birth> <1971-06-28T00:00:00Z> .
Fact: <Tesla, Inc.> <country of origin> <United States> .
Fact: <Tesla, Inc.> <country> <United States> .
Fact: <Tesla, Inc.> <headquarters location> <Austin, Texas> .
Fact: <Tesla, Inc.> <stock exchange> <Nasdaq> .
Fact: <Tesla, Inc.> <stock exchange> <London Stock Exchange> .
Fact: <Tesla, Inc.> <founded by> <Martin Eberhard> .
Fact: <Tesla, Inc.> <founded by> <Marc Tarpenning> .
Fact:
CONTRA:
Fact:
REASONING:
The claim states that Elon Musk founded Tesla. The provided facts explicitly list the founders of Tesla, Inc. as Martin Eberhard and Marc Tarpenning. While the facts confirm that Elon Musk is an employer of Tesla, they do not list him as a founder. In fact, the specific facts regarding the "founded by" relationship contradict the claim by identifying other individuals as the founders. Therefore, the evid

### Generated Facts

In [33]:
for i, triple in enumerate(refactx.get_constrained_states()[0][0].generated_triples):
    print(i, tokenizer.decode(triple), end='\n')

0  <Elon Musk> <employer> <Tesla, Inc.> .
1  <Elon Musk> <date of birth> <1971-06-28T00:00:00Z> .
2  <Tesla, Inc.> <country of origin> <United States> .
3  <Tesla, Inc.> <country> <United States> .
4  <Tesla, Inc.> <headquarters location> <Austin, Texas> .
5  <Tesla, Inc.> <stock exchange> <Nasdaq> .
6  <Tesla, Inc.> <stock exchange> <London Stock Exchange> .
7  <Tesla, Inc.> <founded by> <Martin Eberhard> .
8  <Tesla, Inc.> <founded by> <Marc Tarpenning> .
9 
10 
